# Jour 4 — Évaluation, analyse des erreurs et amélioration (version simple, Kaggle)


## Étape 1 — Importer et configurer

**Problème :** préparer les outils, les paramètres, et savoir **où retrouver** les fichiers du Jour 3.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

CLASSES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

if os.path.exists("/kaggle/working"):
    SPLIT_DIR = Path("/kaggle/working/data/split")
    MODELS_DIR = Path("/kaggle/working/models")
else:
    SPLIT_DIR = Path("data/split")
    MODELS_DIR = Path("models")

# Nouvelle session Kaggle : chercher le split dans les Inputs
if not SPLIT_DIR.exists() and os.path.exists("/kaggle/input"):
    for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
        if dossier.endswith("split") and "train" in sous_dossiers:
            SPLIT_DIR = Path(dossier)
            break

assert SPLIT_DIR.exists(), "Dossier split introuvable : exécutez d'abord le Jour 1."
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def trouver_fichier(nom):
    """Cherche un fichier dans MODELS_DIR, sinon dans les Inputs Kaggle."""
    chemin = MODELS_DIR / nom
    if chemin.exists():
        return chemin
    if os.path.exists("/kaggle/input"):
        for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
            if nom in fichiers:
                return Path(dossier) / nom
    raise FileNotFoundError(nom + " introuvable : exécutez le Jour 3 ou ajoutez sa sortie avec Add Input.")


sns.set_theme()
print("GPU disponible :", tf.config.list_physical_devices("GPU"))

## Étape 2 — Charger les données


In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "train"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

val_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "val"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "test"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_files = test_ds.file_paths    

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# PARTIE A — Évaluer le modèle du Jour 3 (cnn_v1)

## Étape 3 — Prédire sur le jeu de test

**Problème :** pour chaque image de test, on veut connaître la **vraie classe** et la classe **prédite** par le modèle.

**Explication du code :**
- `keras.models.load_model` recharge le modèle sauvegardé au Jour 3.
- `y_true` : la liste des vraies classes (on la récupère dans `test_ds`).
- `model_v1.predict(test_ds)` : pour chaque image, le modèle renvoie **6 probabilités** 
- `np.argmax(probs, axis=1)` choisit, pour chaque image, la classe qui a la **plus grande probabilité** → `y_pred`

In [ ]:
model_v1 = keras.models.load_model(str(trouver_fichier("cnn_v1.keras")))

y_true = np.concatenate([labels.numpy() for images, labels in test_ds])
probs = model_v1.predict(test_ds)
y_pred = np.argmax(probs, axis=1)

print("Nombre d'images de test :", len(y_true))
print("Exemple -> vraie classe :", CLASSES[y_true[0]], "| prédite :", CLASSES[y_pred[0]])

## Étape 4 — Accuracy, Precision, Recall, F1-score

**Problème :** une seule note (l'accuracy) peut cacher qu'une classe est très mal reconnue. On regarde **classe par classe**.

- `classification_report` : tableau avec precision, recall et F1 **pour chaque classe** (`support` = nombre d'images de la classe).

In [ ]:
print("Accuracy globale :", round(accuracy_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

## Étape 5 — Matrice de confusion

**Problème :** savoir **quelles classes sont confondues** entre elles.

**Explication du code :**
- `confusion_matrix(y_true, y_pred)` compte les prédictions pour chaque couple (vraie classe, classe prédite).
- `sns.heatmap` la dessine en couleurs (`annot=True` écrit les nombres dans les cases).
- La boucle à la fin trie les confusions de la **plus fréquente** à la moins fréquente.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Classe prédite")
plt.ylabel("Vraie classe")
plt.title("Matrice de confusion - cnn_v1")
plt.show()

# Les confusions les plus fréquentes
confusions = []
for i in range(6):
    for j in range(6):
        if i != j:
            confusions.append((cm[i, j], CLASSES[i], CLASSES[j]))
confusions.sort(reverse=True)

print("Les 5 confusions les plus fréquentes :")
for nombre, vrai, predit in confusions[:5]:
    print(" ", nombre, "images", vrai, "prises pour", predit)

## Étape 6 — Voir des images mal classées

**Problème :** les chiffres ne disent pas **pourquoi** le modèle se trompe. En regardant les images, on comprend souvent :
image ambiguë (glacier + montagne), photo floue, plusieurs éléments à la fois (rue + bâtiments)...

**Explication du code :**
- `np.where(y_true != y_pred)` donne les **positions** des images où le modèle s'est trompé.
- `np.random.choice` en tire 12 au hasard.
- On ouvre chaque image avec `Image.open(test_files[position])` et on affiche : vraie classe, classe prédite, et la **confiance** du modèle (`probs[...].max()`).

In [ ]:
erreurs = np.where(y_true != y_pred)[0]
print("Nombre d'erreurs :", len(erreurs), "sur", len(y_true), "images")

np.random.seed(SEED)
choix = np.random.choice(erreurs, size=min(12, len(erreurs)), replace=False)

plt.figure(figsize=(14, 10))
for k, position in enumerate(choix):
    plt.subplot(3, 4, k + 1)
    plt.imshow(Image.open(test_files[position]))
    vrai = CLASSES[y_true[position]]
    predit = CLASSES[y_pred[position]]
    confiance = probs[position].max()
    plt.title("Vrai : " + vrai + "\nPrédit : " + predit + " (" + str(round(confiance * 100)) + " %)", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Étape 7 — Overfitting / underfitting du modèle v1


In [ ]:
hist_v1 = pd.read_csv(trouver_fichier("history_cnn_v1.csv"))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(hist_v1["loss"], label="train")
plt.plot(hist_v1["val_loss"], label="validation")
plt.title("Loss - cnn_v1")
plt.xlabel("époque")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist_v1["accuracy"], label="train")
plt.plot(hist_v1["val_accuracy"], label="validation")
plt.title("Accuracy - cnn_v1")
plt.xlabel("époque")
plt.legend()

plt.show()

# PARTIE B — Tester 2 autres configurations

**Problème :** le premier modèle est-il le meilleur possible ? On teste des variantes pour **comparer** objectivement.

`cnn_v1` : (déjà entraîné au Jour 3) 3 blocs, dropout 0.3, **avec** augmentation 
`cnn_v2_sans_aug`: **sans** Data Augmentation , Quel est l'impact de l'augmentation ? 
`cnn_v3_plus_profond` : **4 blocs** (256 filtres en plus), dropout **0.5**  Un modèle plus gros est-il meilleur ? 
## Étape 8 — Une fonction pour construire le modèle

**Problème :** construire 2 modèles presque identiques sans recopier 20 lignes à chaque fois.

In [ ]:
def construire_cnn(filtres, dropout, augmentation):
    model = keras.Sequential()
    model.add(keras.Input(shape=(150, 150, 3)))

    if augmentation:
        model.add(keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomContrast(0.1),
        ]))

    model.add(layers.Rescaling(1.0 / 255))

    for nb in filtres:                      
        model.add(layers.Conv2D(nb, (3, 3), activation="relu", padding="same"))
        model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(6, activation="softmax"))
    return model


# On range ici les modèles et leurs historiques (pour la comparaison finale)
modeles = {"cnn_v1": model_v1}
historiques = {"cnn_v1": hist_v1}

## Étape 9 — Configuration 2 : sans Data Augmentation


In [ ]:
model_v2 = construire_cnn(filtres=[32, 64, 128], dropout=0.3, augmentation=False)

model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

arret_precoce = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduire_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

history_v2 = model_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[arret_precoce, reduire_lr],
)

model_v2.save(str(MODELS_DIR / "cnn_v2_sans_aug.keras"))
modeles["cnn_v2_sans_aug"] = model_v2
historiques["cnn_v2_sans_aug"] = pd.DataFrame(history_v2.history)

## Étape 10 — Configuration 3 : modèle plus profond, dropout 0.5

**Problème :** un modèle avec **plus de filtres et plus de couches** apprend-il mieux ? Un dropout plus fort limite-t-il l'overfitting ?

In [ ]:
model_v3 = construire_cnn(filtres=[32, 64, 128, 256], dropout=0.5, augmentation=True)

model_v3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

arret_precoce = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduire_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

history_v3 = model_v3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[arret_precoce, reduire_lr],
)

model_v3.save(str(MODELS_DIR / "cnn_v3_plus_profond.keras"))
modeles["cnn_v3_plus_profond"] = model_v3
historiques["cnn_v3_plus_profond"] = pd.DataFrame(history_v3.history)

# PARTIE C — Comparer et choisir le modèle final

## Étape 11 — Tableau de comparaison

**Problème :** choisir le meilleur modèle de façon **objective**, avec des chiffres.

In [ ]:
lignes = []

for nom in modeles:
    hist = historiques[nom]
    meilleure = hist["val_loss"].idxmin()

    probs_nom = modeles[nom].predict(test_ds, verbose=0)
    y_pred_nom = np.argmax(probs_nom, axis=1)

    lignes.append({
        "modele": nom,
        "epoques": len(hist),
        "acc_train": hist.loc[meilleure, "accuracy"],
        "acc_val": hist.loc[meilleure, "val_accuracy"],
        "ecart": hist.loc[meilleure, "accuracy"] - hist.loc[meilleure, "val_accuracy"],
        "acc_test": accuracy_score(y_true, y_pred_nom),
        "f1_test": f1_score(y_true, y_pred_nom, average="macro"),
    })

tableau = pd.DataFrame(lignes).set_index("modele").round(3)
print(tableau)

## Étape 12 — Graphiques de comparaison

**Explication du code :**
- Le 1er graphique compare l'accuracy de **validation** et de **test** de chaque modèle.
- Ensuite, pour chaque modèle, on trace ses courbes Loss et Accuracy (les mêmes que celles du Jour 3) pour comparer les **écarts train / validation**.

In [ ]:
tableau[["acc_val", "acc_test"]].plot(kind="bar", figsize=(8, 4), ylim=(0, 1))
plt.title("Comparaison des modèles")
plt.ylabel("accuracy")
plt.xticks(rotation=15)
plt.show()

for nom in historiques:
    hist = historiques[nom]
    plt.figure(figsize=(12, 3.5))

    plt.subplot(1, 2, 1)
    plt.plot(hist["loss"], label="train")
    plt.plot(hist["val_loss"], label="validation")
    plt.title("Loss - " + nom)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist["accuracy"], label="train")
    plt.plot(hist["val_accuracy"], label="validation")
    plt.title("Accuracy - " + nom)
    plt.legend()

    plt.show()

## Étape 13 — Choisir et sauvegarder le modèle final

**Problème :** garder le meilleur modèle sous le nom `final_model.keras` pour le Jour 5.

**Explication du code :**
- `tableau["acc_val"].idxmax()` donne le **nom du modèle** qui a la meilleure accuracy de **validation**.
- On sauvegarde ce modèle sous le nom `final_model.keras`, et son nom dans un petit fichier texte.

In [ ]:
meilleur_nom = tableau["acc_val"].idxmax()
print("Modèle retenu :", meilleur_nom)
print(tableau.loc[meilleur_nom])

modeles[meilleur_nom].save(str(MODELS_DIR / "final_model.keras"))
with open(MODELS_DIR / "final_model_name.txt", "w") as fichier:
    fichier.write(meilleur_nom)

print("Fichiers dans models :", os.listdir(MODELS_DIR))

## Étape 14 : sauvegarde comparaison.csv

In [ ]:
tableau.to_csv(MODELS_DIR / "comparaison.csv")
for nom in historiques:
    historiques[nom].to_csv(MODELS_DIR / ("history_" + nom + ".csv"), index=False)